# Phase 2 — Detection analysis (decision gate)

Run after **14-21 days** of `agent.cli detect` on the VPS.

Decision rule: **PROCEED_TO_LIVE** if median day has ≥3 gaps with `net_gap_pct >= 6%` and simulated P&L positive; else **CONCLUDE_PROJECT**.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

DB_PATH = "../data/agent.db"
THRESHOLD = 6.0
engine = create_engine(f"sqlite:///{DB_PATH}")
gaps = pd.read_sql(text("SELECT * FROM cross_chain_gaps"), engine)
snapshots = pd.read_sql(text("SELECT * FROM quote_snapshots"), engine)
print(f"snapshots: {len(snapshots):,}  gaps: {len(gaps):,}")

In [ ]:
gaps["day"] = pd.to_datetime(gaps["ts"]).dt.date
gaps["actionable"] = gaps["net_gap_pct"] >= THRESHOLD
daily = gaps.groupby("day")["actionable"].sum()
print(daily.describe())
gaps.groupby(["chain_a", "chain_b"])["net_gap_pct"].describe()

In [ ]:
median_actionable = daily.median() if len(daily) else 0
if median_actionable >= 3:
    print(">>> PROCEED_TO_LIVE <<<")
else:
    print(">>> CONCLUDE_PROJECT <<<")